# SupplyMind AI — Random Forest

Model selection is performed on validation data only.

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
from supplymind.features.predictions.ml.evaluation import (
    choose_threshold,
    evaluate_probabilities,
    positive_class_probability,
)
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.reporting import (
    save_evaluation_plots,
    save_feature_importance,
    save_json,
)
from supplymind.features.predictions.ml.training import (
    build_random_forest,
    fit_pipeline,
)
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Prepare identical model data
# -------------------

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

In [4]:
# -------------------
# Build preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=False,
)

In [5]:
# -------------------
# Train model
# -------------------

estimator = build_random_forest()
model = fit_pipeline(
    preprocessor,
    estimator,
    data.X_train,
    data.y_train,
)

In [6]:
# -------------------
# Validation probabilities
# -------------------

validation_probability = positive_class_probability(
    model,
    data.X_validation,
)

threshold, threshold_search = choose_threshold(
    data.y_validation,
    validation_probability,
)

print("Selected threshold:", threshold)
threshold_search.sort_values(
    ["f1", "recall"],
    ascending=False,
).head(10)

Selected threshold: 0.3000000000000001


,accuracy,precision,recall,f1,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,threshold
10,0.582515,0.581378,0.988412,0.732125,0.738214,0.83387,280,9581,156,13306,0.30
4,0.577241,0.577223,1.000000,0.731949,0.738214,0.83387,1,9860,0,13462,0.24
0,0.577198,0.577198,1.000000,0.731929,0.738214,0.83387,0,9861,0,13462,0.20
1,0.577198,0.577198,1.000000,0.731929,0.738214,0.83387,0,9861,0,13462,0.21
2,0.577198,0.577198,1.000000,0.731929,0.738214,0.83387,0,9861,0,13462,0.22
3,0.577198,0.577198,1.000000,0.731929,0.738214,0.83387,0,9861,0,13462,0.23
5,0.577113,0.577169,0.999777,0.731845,0.738214,0.83387,1,9860,3,13459,0.25
7,0.577284,0.577367,0.998663,0.731706,0.738214,0.83387,20,9841,18,13444,0.27
6,0.577027,0.577172,0.999183,0.731689,0.738214,0.83387,7,9854,11,13451,0.26
9,0.579042,0.578868,0.993389,0.731485,0.738214,0.83387,132,9729,89,13373,0.29


In [7]:
# -------------------
# Validation metrics
# -------------------

metrics = evaluate_probabilities(
    data.y_validation,
    validation_probability,
    threshold=threshold,
)

metrics.to_dict()

{'accuracy': 0.5825151138361274,
 'precision': 0.5813780748896754,
 'recall': 0.9884118258802556,
 'f1': 0.732124680183774,
 'roc_auc': 0.7382135227425288,
 'average_precision': 0.8338696672202058,
 'true_negative': 280,
 'false_positive': 9581,
 'false_negative': 156,
 'true_positive': 13306,
 'threshold': 0.3000000000000001}

In [8]:
# -------------------
# Save candidate reports
# -------------------

MODEL_NAME = "random_forest"
REPORT_DIR = REPORT_ROOT / "models" / MODEL_NAME

save_json(
    metrics.to_dict(),
    REPORT_DIR / "validation_metrics.json",
)
threshold_search.to_csv(
    REPORT_DIR / "threshold_search.csv",
    index=False,
)
save_evaluation_plots(
    data.y_validation,
    validation_probability,
    threshold,
    REPORT_DIR,
    "validation",
)
save_feature_importance(
    model,
    REPORT_DIR / "feature_importance",
)

save_model_artifact(
    model,
    {
        "model_name": MODEL_NAME,
        "model_version": "0.1.0-candidate",
        "threshold": threshold,
        "validation_metrics": metrics.to_dict(),
    },
    MODEL_ROOT / "candidates" / MODEL_NAME,
)